# Beat the Press — Press Resistance Index (PRI), version 3

**What changed from the previous version**
1. **More data:** 3 → 8 competitions (163 → 383 matches).
2. **Carry label fixed:** if the carrier gets *fouled*, his team keeps the ball, so the carry is **retained**.
3. **Model fixed:** `class_weight="balanced"` removed (it made the probabilities wrong), `position` removed from the model, better features, XGBoost.
4. **Extended OE (OE⁺):** OE plus a value-weighted version (RVA) that counts more when the ball is kept in dangerous areas.
5. **Fairer threat/progression:** only counted when the ball is actually kept.

## 1. Setup

In [ ]:
import os, json, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, brier_score_loss
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
os.makedirs("results", exist_ok=True)   # figures and tables are saved here
pd.set_option("display.max_columns", 50)

## 2. Data Preparation

### Downloading the data
We use 8 competitions from StatsBomb's free open data. Files are cached, so re-runs are fast.

In [ ]:
BASE = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"
CACHE = "sb_data"
os.makedirs(CACHE, exist_ok=True)

# competition_id, season_id, display name
COMPETITIONS = {
    "WC2022":     (43,   106, "FIFA World Cup 2022"),
    "WC2018":     (43,   3,   "FIFA World Cup 2018"),
    "LaLiga2021": (11,   90,  "La Liga 2020/2021"),
    "Euro2020":   (55,   43,  "UEFA Euro 2020"),
    "Euro2024":   (55,   282, "UEFA Euro 2024"),
    "Copa2024":   (223,  282, "Copa America 2024"),
    "AFCON2023":  (1267, 107, "AFCON 2023"),
    "BL2324":     (9,    281, "Bundesliga 2023/2024"),
}

def get_json(url, cache_name):
    path = os.path.join(CACHE, cache_name)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    import urllib.request
    with urllib.request.urlopen(url) as r:
        data = json.loads(r.read().decode("utf-8"))
    with open(path, "w") as f:
        json.dump(data, f)
    return data

def fetch_all():
    match_index = {}
    for name, (comp_id, season_id, _) in COMPETITIONS.items():
        matches = get_json(f"{BASE}/matches/{comp_id}/{season_id}.json", f"matches_{comp_id}_{season_id}.json")
        match_index[name] = matches
        for m in matches:
            mid = m["match_id"]
            try:
                get_json(f"{BASE}/events/{mid}.json", f"events_{mid}.json")
            except Exception as e:
                print("FAILED", mid, e)
        print(name, "->", len(matches), "matches")
    return match_index

t0 = time.time()
match_index = fetch_all()
print(f"Data ready in {time.time()-t0:.0f}s")

### Building one events table
We read every event of every match and keep only the columns we need.

In [ ]:
def safe(d, *keys):
    for k in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
    return d

def to_seconds(ts):
    h, m, s = ts.split(":")
    return int(h)*3600 + int(m)*60 + float(s)

def extract_match(mid, comp_name):
    with open(os.path.join(CACHE, f"events_{mid}.json")) as f:
        events = json.load(f)
    rows = []
    for e in events:
        loc = e.get("location") or [None, None]
        p_end = safe(e, "pass", "end_location") or [None, None]
        c_end = safe(e, "carry", "end_location") or [None, None]
        rows.append({
            "match_id": mid, "id": e["id"], "index": e["index"], "period": e["period"],
            "minute": e["minute"], "t": to_seconds(e["timestamp"]),
            "team": safe(e, "team", "name"), "player": safe(e, "player", "name"),
            "position": safe(e, "position", "name"), "type": safe(e, "type", "name"),
            "x": loc[0], "y": loc[1],
            "under_pressure": e.get("under_pressure", False), "counterpress": e.get("counterpress", False),
            "possession": e.get("possession"), "play_pattern": safe(e, "play_pattern", "name"),
            "pass_end_x": p_end[0], "pass_end_y": p_end[1],
            "pass_outcome": safe(e, "pass", "outcome", "name"), "pass_angle": safe(e, "pass", "angle"),
            "pass_height": safe(e, "pass", "height", "name"), "pass_body": safe(e, "pass", "body_part", "name"),
            "pass_type": safe(e, "pass", "type", "name"), "pass_technique": safe(e, "pass", "technique", "name"),
            "pass_cross": bool(safe(e, "pass", "cross")), "pass_switch": bool(safe(e, "pass", "switch")),
            "pass_through": bool(safe(e, "pass", "through_ball")),
            "carry_end_x": c_end[0], "carry_end_y": c_end[1],
            "shot_xg": safe(e, "shot", "statsbomb_xg"), "duel_outcome": safe(e, "duel", "outcome", "name"),
            "competition_name": comp_name,
        })
    return pd.DataFrame(rows).sort_values(["period", "index"]).reset_index(drop=True)

### Fixing the carry retention label
For each Carry we look at the next event that actually matters (we skip things like Pressure or Ball Receipt):
- **Same team** plays on (pass, carry, shot...) → **retained**
- **Same team** is Dispossessed / Miscontrol / Error / commits a Foul → **lost**
- **Opponent** commits a Foul → carrier was fouled, team keeps the ball → **retained** *(this was wrong before)*
- **Opponent** loses a tackle (Duel lost) → keep looking at the next event
- **Opponent** wins the ball (recovery, interception, clearance, pass...) → **lost**

We also keep the **old rule** so we can compare the old and new PRI.

In [ ]:
SKIP_TYPES = {"Pressure", "Ball Receipt*", "Camera On", "Camera off", "Starting XI", "Half Start", "Half End",
              "Tactical Shift", "Injury Stoppage", "Substitution", "Player On", "Player Off",
              "Referee Ball-Drop", "Dribbled Past", "Shield", "Bad Behaviour", "Offside",
              "Own Goal For", "Own Goal Against"}
OWN_TEAM_LOSS = {"Dispossessed", "Miscontrol", "Error", "Foul Committed"}
OPPONENT_KEEP_LOOKING = {"Duel", "Block", "Pressure", "50/50", "Dribbled Past"}

def label_carries(mdf):
    types, teams = mdf["type"].values, mdf["team"].values
    periods, possessions = mdf["period"].values, mdf["possession"].values
    duel = mdf["duel_outcome"].values
    n = len(mdf)
    new_label, old_label = np.full(n, np.nan), np.full(n, np.nan)

    for i in np.where(types == "Carry")[0]:
        # ---- OLD rule (previous version, for comparison) ----
        j = i + 1
        while j < n and types[j] in SKIP_TYPES and possessions[j] == possessions[i] and periods[j] == periods[i]:
            j += 1
        if j >= n or possessions[j] != possessions[i] or periods[j] != periods[i]:
            old_label[i] = 0
        elif types[j] == "50/50":
            pass
        elif types[j] in OWN_TEAM_LOSS or teams[j] != teams[i]:
            old_label[i] = 0
        else:
            old_label[i] = 1

        # ---- NEW rule ----
        j = i + 1
        while j < n and j < i + 10 and periods[j] == periods[i]:
            t = types[j]
            if t in SKIP_TYPES:
                j += 1; continue
            if teams[j] == teams[i]:                       # own team
                if t in {"Duel", "50/50"}:
                    j += 1; continue
                new_label[i] = 0 if t in OWN_TEAM_LOSS else 1
                break
            else:                                          # opponent
                if t == "Foul Committed":
                    new_label[i] = 1; break                # carrier was fouled
                if t in OPPONENT_KEEP_LOOKING or (t == "Duel" and duel[j] in ("Lost In Play", "Lost Out")):
                    j += 1; continue
                new_label[i] = 0; break                    # opponent has the ball
    mdf["carry_retained_new"] = new_label
    mdf["carry_retained_old"] = old_label
    return mdf

### Adding pressure context
These describe the situation **before** the action, so they are safe to use in the model:
- `opp_pressures_3s`: how many times the opponent pressed this team in the last 3 seconds
- `prev_type`: the event just before (e.g. a Ball Receipt, a Duel)
- `poss_age`: seconds since the possession started

In [ ]:
def add_context(mdf):
    pressures = mdf[mdf["type"] == "Pressure"]
    counts = np.zeros(len(mdf))
    for (period, team), idx in mdf.groupby(["period", "team"]).groups.items():
        opp_times = np.sort(pressures[(pressures["period"] == period) & (pressures["team"] != team)]["t"].values)
        t = mdf.loc[idx, "t"].values
        counts[idx] = np.searchsorted(opp_times, t, "right") - np.searchsorted(opp_times, t - 3.0, "left")
    mdf["opp_pressures_3s"] = counts

    on_ball = mdf["type"].isin(["Pass", "Carry", "Ball Receipt*", "Ball Recovery", "Duel", "Dribble", "Interception"])
    mdf["prev_type"] = mdf["type"].where(on_ball).shift(1)
    mdf["poss_age"] = (mdf["t"] - mdf.groupby(["period", "possession"])["t"].transform("min")).clip(0, 120)
    return mdf

def build_all():
    all_events = []
    for name, (_, _, comp_name) in COMPETITIONS.items():
        for m in match_index[name]:
            mdf = extract_match(m["match_id"], comp_name)
            mdf = label_carries(mdf)
            mdf = add_context(mdf)
            all_events.append(mdf[mdf["type"].isin(["Pass", "Carry", "Shot"])])   # keep only what we need
        print(name, "done")
    return pd.concat(all_events, ignore_index=True)

events = build_all()
print("events (Pass + Carry + Shot):", events.shape)

In [ ]:
carries = events[events["type"] == "Carry"]
up = carries["under_pressure"] == True
print(f"Carry retention UNDER pressure  - old rule: {carries.loc[up,'carry_retained_old'].mean()*100:.1f}%"
      f"   new rule: {carries.loc[up,'carry_retained_new'].mean()*100:.1f}%")
print(f"Carry retention with NO pressure - old rule: {carries.loc[~up,'carry_retained_old'].mean()*100:.1f}%"
      f"   new rule: {carries.loc[~up,'carry_retained_new'].mean()*100:.1f}%")

### Dataset description

In [ ]:
pc = events[events["type"].isin(["Pass", "Carry"])]
desc = pc.groupby("competition_name").agg(
    Matches=("match_id", "nunique"), PassCarry_events=("id", "count"),
    Under_pressure_pct=("under_pressure", "mean"), Players=("player", "nunique"))
desc["Under_pressure_pct"] = (desc["Under_pressure_pct"] * 100).round(1)
desc.loc["TOTAL"] = [pc["match_id"].nunique(), len(pc), round(pc["under_pressure"].mean()*100, 1), pc["player"].nunique()]
desc

## 3. Exploratory Data Analysis

One `retained` column: passes are retained if completed, carries use the new label.
Passes with outcome "Unknown" or "Injury Clearance" are removed.

In [ ]:
events = events[~events["pass_outcome"].isin(["Unknown", "Injury Clearance"])].copy()
is_pass, is_carry = events["type"] == "Pass", events["type"] == "Carry"

events["retained"] = np.where(is_pass, events["pass_outcome"].isna(), events["carry_retained_new"])
events["retained_old"] = np.where(is_pass, events["pass_outcome"].isna(), events["carry_retained_old"])
events["end_x"] = np.where(is_pass, events["pass_end_x"], events["carry_end_x"])
events["end_y"] = np.where(is_pass, events["pass_end_y"], events["carry_end_y"])

pass_carry = events[(is_pass | is_carry) & events["retained"].notna() & events["retained_old"].notna()
                    & events["x"].notna() & events["end_x"].notna()].copy()
pass_carry["forward_distance"] = pass_carry["end_x"] - pass_carry["x"]
print(f"Pass+Carry events: {len(pass_carry):,}   under pressure: {pass_carry['under_pressure'].mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pass_carry["competition_name"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Pass + Carry events by competition"); axes[0].tick_params(axis="x", rotation=30)
(pass_carry.groupby("competition_name")["under_pressure"].mean()*100).sort_values().plot(kind="bar", ax=axes[1], color="indianred")
axes[1].set_title("% of Pass/Carry events under pressure"); axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
up_pc = pass_carry[pass_carry["under_pressure"] == True]
sep = up_pc.groupby("type").agg(n=("id", "count"), retention_pct=("retained", "mean"),
                                avg_forward_distance=("forward_distance", "mean"))
sep["retention_pct"] = (sep["retention_pct"] * 100).round(1)
print("Pass vs Carry UNDER pressure:")
sep.round(2)

In [ ]:
share = pass_carry.groupby("position")["under_pressure"].mean().sort_values() * 100
share.plot(kind="barh", figsize=(9, 8), color="crimson")
plt.title("Share of Pass/Carry actions under pressure, by position"); plt.xlabel("%")
plt.tight_layout(); plt.show()

## 4. Methodology

### Expected Threat (xT)
We split the pitch into a 12 × 8 grid and compute xT the standard way (Karun Singh):
the value of a zone = chance of shooting × chance of scoring + chance of moving × value of where the ball goes next.

In [ ]:
N_X, N_Y = 12, 8

def zone_of(x, y):
    zx = np.clip((np.nan_to_num(x) // 10).astype(int), 0, N_X - 1)
    zy = np.clip((np.nan_to_num(y) // 10).astype(int), 0, N_Y - 1)
    return zx * N_Y + zy

def build_xT(pass_carry, shots):
    start = zone_of(pass_carry["x"].values, pass_carry["y"].values)
    end = zone_of(pass_carry["end_x"].values, pass_carry["end_y"].values)
    ok = pass_carry["retained"].values == 1
    shot_zone = zone_of(shots["x"].values, shots["y"].values)

    n_moves = np.bincount(start, minlength=96).astype(float)
    n_shots = np.bincount(shot_zone, minlength=96).astype(float)
    goal_prob = np.bincount(shot_zone, weights=shots["shot_xg"].fillna(0).values, minlength=96) / np.maximum(n_shots, 1)
    shoot_prob = n_shots / np.maximum(n_moves + n_shots, 1)
    move_prob = n_moves / np.maximum(n_moves + n_shots, 1)

    transition = np.zeros((96, 96))
    np.add.at(transition, (start[ok], end[ok]), 1)       # only successful moves
    transition = transition / np.maximum(n_moves[:, None], 1)

    xT = np.zeros(96)
    for _ in range(100):
        xT = shoot_prob * goal_prob + move_prob * (transition @ xT)
    return xT

xT = build_xT(pass_carry, events[events["type"] == "Shot"])

plt.figure(figsize=(10, 5))
sns.heatmap(xT.reshape(N_X, N_Y).T, cmap="Reds", cbar_kws={"label": "xT"})
plt.gca().invert_yaxis()
plt.title("xT grid (own goal on the left, opponent goal on the right)")
plt.tight_layout(); plt.savefig("results/xT_grid.png", dpi=150); plt.show()

### Features for the pressured actions

In [ ]:
POS_GROUP = {
 "Goalkeeper":"GK","Center Back":"CB","Left Center Back":"CB","Right Center Back":"CB",
 "Left Back":"FB","Right Back":"FB","Left Wing Back":"FB","Right Wing Back":"FB",
 "Center Defensive Midfield":"DM","Left Defensive Midfield":"DM","Right Defensive Midfield":"DM",
 "Center Midfield":"CM","Left Center Midfield":"CM","Right Center Midfield":"CM",
 "Center Attacking Midfield":"AM","Left Attacking Midfield":"AM","Right Attacking Midfield":"AM",
 "Left Midfield":"WM","Right Midfield":"WM","Left Wing":"WING","Right Wing":"WING",
 "Center Forward":"FWD","Left Center Forward":"FWD","Right Center Forward":"FWD","Secondary Striker":"FWD",
}
ORDER = ["GK","CB","FB","DM","CM","AM","WM","WING","FWD"]

pressured = pass_carry[(pass_carry["under_pressure"] == True) & pass_carry["position"].notna()].copy()
pressured["pos_group"] = pressured["position"].map(POS_GROUP)

# xT at start, at end, and for the OPPONENT if they win the ball at the end spot
pressured["xT_start"] = xT[zone_of(pressured["x"].values, pressured["y"].values)]
pressured["xT_end"] = xT[zone_of(pressured["end_x"].values, pressured["end_y"].values)]
pressured["xT_opponent"] = xT[zone_of(120 - pressured["end_x"].values, 80 - pressured["end_y"].values)]

# simple geometry features
pressured["forward_distance_capped"] = pressured["forward_distance"].clip(upper=40)
pressured["length"] = np.hypot(pressured["end_x"] - pressured["x"], pressured["end_y"] - pressured["y"])
pressured["dist_to_goal"] = np.hypot(120 - pressured["x"], 40 - pressured["y"])
pressured["angle_to_goal"] = np.abs(np.arctan2(40 - pressured["y"], 120 - pressured["x"]))
pressured["pass_angle_sin"] = np.sin(pressured["pass_angle"].fillna(0))
pressured["pass_angle_cos"] = np.cos(pressured["pass_angle"].fillna(0))

for c in ["pass_height", "pass_body", "pass_type", "pass_technique", "prev_type", "play_pattern"]:
    pressured[c] = pressured[c].fillna("none")
for c in ["pass_cross", "pass_switch", "pass_through", "counterpress"]:
    pressured[c] = pressured[c].astype(int)

print(f"Pressured actions: {len(pressured):,}")

### Expected retention model
Predicts P(retained) for each pressured action from **how hard the situation was**.
`position` is **not** used — the model should judge the situation, not the player's role.

In [ ]:
NUMERIC = ["x", "y", "dist_to_goal", "angle_to_goal", "pass_angle_sin", "pass_angle_cos",
           "opp_pressures_3s", "poss_age", "minute", "pass_cross", "pass_switch", "pass_through",
           "counterpress", "end_x", "end_y", "forward_distance_capped", "length"]
CATEGORICAL = ["type", "pass_height", "pass_body", "pass_type", "pass_technique", "prev_type", "play_pattern"]

def cross_validate(df, numeric, categorical, label, model):
    """5-fold CV, grouped by match. Returns out-of-fold predictions, mean AUC, Brier score."""
    X, y = df[numeric + categorical], df[label].astype(int).values
    oof, aucs = np.zeros(len(df)), []
    for tr, te in GroupKFold(n_splits=5).split(X, y, df["match_id"]):
        pipe = Pipeline([
            ("prep", ColumnTransformer([("num", StandardScaler(), numeric),
                                        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30), categorical)])),
            ("clf", model)])
        pipe.fit(X.iloc[tr], y[tr])
        oof[te] = pipe.predict_proba(X.iloc[te])[:, 1]
        aucs.append(roc_auc_score(y[te], oof[te]))
    return oof, np.mean(aucs), brier_score_loss(y, oof)

xgb = XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                    random_state=42)   # fixed seed -> same numbers on every run
pressured["p_hat"], auc_new, brier_new = cross_validate(pressured, NUMERIC, CATEGORICAL, "retained", xgb)

# the OLD model, for comparison (old labels, old features, balanced logistic regression)
pressured["pass_length0"] = np.where(pressured["type"] == "Pass", pressured["length"], 0)
old_lr = LogisticRegression(max_iter=3000, class_weight="balanced")
pressured["p_hat_old"], auc_old, brier_old = cross_validate(
    pressured, ["x", "y", "forward_distance_capped", "pass_length0"], ["type", "position"], "retained_old", old_lr)

base_rate = pressured["retained"].mean()
model_table = pd.DataFrame({
    "Model": ["Old: logistic regression (balanced)", "New: XGBoost + pressure context"],
    "ROC-AUC": [auc_old, auc_new],
    "Brier": [brier_old, brier_new],
})
model_table.round(3).to_csv("results/model_comparison.csv", index=False)
model_table.round(3)

*A Brier score is only useful if it beats always guessing the average. The old model did not (see print below) — that was the `class_weight="balanced"` bug.*

In [ ]:
print(f"Brier if we always guess the average (old labels): {pressured['retained_old'].mean()*(1-pressured['retained_old'].mean()):.3f}")
print(f"Brier if we always guess the average (new labels): {base_rate*(1-base_rate):.3f}")

### The PRI components
For each pressured action:
- **OE** = retained − p̂ (did the player keep the ball more than expected?)
- **RVA (extended OE)** = OE × (xT_end + xT_opponent) × 100.
  Keeping the ball in a dangerous spot (own box, or near their goal) is worth more than in midfield.
- **Threat** = xT gained, **only if the ball was kept**
- **Progression** = moved the ball ≥ 10 units forward, **only if the ball was kept**

**OE⁺** = ½ · z(OE) + ½ · z(RVA)

$$\text{PRI} = 0.4 \cdot Z(OE^+) + 0.3 \cdot Z(\text{Threat}) + 0.3 \cdot Z(\text{Progression})$$

In [ ]:
pressured["OE"] = pressured["retained"] - pressured["p_hat"]
pressured["RVA"] = 100 * pressured["OE"] * (pressured["xT_end"] + pressured["xT_opponent"])
pressured["threat"] = 100 * pressured["retained"] * (pressured["xT_end"] - pressured["xT_start"])
pressured["progression"] = pressured["retained"] * (pressured["forward_distance"] >= 10)

# old versions of the components (as in the previous paper)
pressured["OE_old"] = pressured["retained_old"] - pressured["p_hat_old"]
pressured["threat_old"] = 100 * (pressured["xT_end"] - pressured["xT_start"])      # counted even when lost
pressured["progression_old"] = (pressured["forward_distance"] >= 10).astype(float)

In [ ]:
def zscore(s): return (s - s.mean()) / s.std(ddof=0)
def shrink(x, n, k): return (n / (n + k)) * x + (k / (n + k)) * x.mean()

def build_pri(df, min_actions=50, weights=(0.4, 0.3, 0.3), k=20, old=False):
    cols = ["OE_old", "threat_old", "progression_old"] if old else ["OE", "RVA", "threat", "progression"]
    g = df.groupby("player").agg(
        team=("team", lambda s: s.mode()[0]), pos_group=("pos_group", lambda s: s.mode()[0]),
        pressured_actions=("id", "count"), retention=("retained", "mean"),
        **{c: (c, "mean") for c in cols}).reset_index()
    g = g[g["pressured_actions"] >= min_actions].copy()
    for c in cols:
        g[c] = shrink(g[c], g["pressured_actions"], k)              # stabilise small samples

    wR, wT, wP = weights
    if old:
        g["PRI"] = wR*zscore(g["OE_old"]) + wT*zscore(g["threat_old"]) + wP*zscore(g["progression_old"])
    else:
        g["OE_plus"] = zscore(0.5*zscore(g["OE"]) + 0.5*zscore(g["RVA"]))
        g["PRI"] = wR*g["OE_plus"] + wT*zscore(g["threat"]) + wP*zscore(g["progression"])
    return g.sort_values("PRI", ascending=False).reset_index(drop=True)

outfield = pressured[pressured["pos_group"] != "GK"]      # goalkeepers are ranked separately
pri = build_pri(outfield)
pri_old = build_pri(pressured, old=True)
print("Players with >= 50 pressured actions (outfield):", len(pri))
pri.round(4).to_csv("results/pri_v3_outfield.csv", index=False)

## 5. Results

### Top 15 players by PRI v3

In [ ]:
top15 = pri.head(15)[["player", "team", "pos_group", "pressured_actions", "retention", "OE", "RVA", "threat", "progression", "PRI"]].copy()
top15["retention"] = (top15["retention"] * 100).round(1)
top15.index = range(1, 16)
top15.round(3)

In [ ]:
plot_df = pri.head(15).iloc[::-1]
plt.figure(figsize=(9, 7))
plt.barh(plot_df["player"] + " (" + plot_df["pos_group"] + ")", plot_df["PRI"], color="seagreen")
plt.title("Top 15 outfield players by PRI v3"); plt.xlabel("PRI")
plt.tight_layout(); plt.savefig("results/top15_pri.png", dpi=150); plt.show()

### Do defenders get a fair PRI now?

In [ ]:
by_group = pd.DataFrame({
    "Old PRI": pri_old.groupby("pos_group")["PRI"].mean(),
    "New PRI (v3)": pri.groupby("pos_group")["PRI"].mean(),
}).reindex(ORDER)

by_group.plot(kind="bar", figsize=(11, 5), color=["#C44E52", "#4C72B0"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Average PRI by position group: old vs new"); plt.ylabel("Average PRI"); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig("results/pri_by_position.png", dpi=150); plt.show()
by_group.round(2).to_csv("results/pri_by_position.csv")
by_group.round(2)

In [ ]:
print("Top 10 centre-backs:")
pri[pri["pos_group"] == "CB"].head(10)[["player", "team", "pressured_actions", "OE", "RVA", "PRI"]].round(3)

## 6. Robustness & Validation

### Changing the weights

In [ ]:
base = pri.set_index("player")["PRI"]
for w in [(1/3, 1/3, 1/3), (0.5, 0.25, 0.25), (0.25, 0.5, 0.25), (0.25, 0.25, 0.5)]:
    other = build_pri(outfield, weights=w).set_index("player")["PRI"]
    print(f"weights {np.round(w, 2)} -> Spearman rho vs base: {spearmanr(base, other.loc[base.index])[0]:.3f}")

### Changing the minimum number of actions

In [ ]:
for thr in [30, 50, 70, 100]:
    p = build_pri(outfield, min_actions=thr)
    print(f"min_actions={thr:3d}  ->  players={len(p):4d}   top 3: {list(p.head(3)['player'])}")

### Split-half reliability
Split the matches randomly in two halves, compute PRI in each, and check if players keep a similar rank.
Higher = more stable. (Tournament samples are small, so expect modest values.)

In [ ]:
matches = pressured["match_id"].unique()
rng = np.random.default_rng(0)
rhos = []
for _ in range(5):
    half = rng.choice(matches, len(matches) // 2, replace=False)
    a = build_pri(outfield[outfield["match_id"].isin(half)], min_actions=25).set_index("player")["PRI"]
    b = build_pri(outfield[~outfield["match_id"].isin(half)], min_actions=25).set_index("player")["PRI"]
    common = a.index.intersection(b.index)
    rhos.append(spearmanr(a[common], b[common])[0])
print(f"Split-half reliability of PRI v3: {np.mean(rhos):.3f}")

### Does the result depend on end-location features?
For a lost pass, the end location is where the ball was stopped, which slightly "leaks" the outcome.
So we refit the model **without** end location / length and compare.

In [ ]:
PRE_ACTION = [c for c in NUMERIC if c not in ["end_x", "end_y", "forward_distance_capped", "length"]]
pre = outfield.copy()
pre["p_hat"], auc_pre, _ = cross_validate(pre, PRE_ACTION, CATEGORICAL, "retained", xgb)
pre["OE"] = pre["retained"] - pre["p_hat"]
pre["RVA"] = 100 * pre["OE"] * (pre["xT_end"] + pre["xT_opponent"])
pri_pre = build_pri(pre).set_index("player")["PRI"]
print(f"AUC without end-location features: {auc_pre:.3f}")
print(f"Spearman rho of PRI (with vs without): {spearmanr(base, pri_pre.loc[base.index])[0]:.3f}")

### Leave-one-competition-out
Train on 7 competitions, test on the 8th. If AUC stays high, the model generalises.

In [ ]:
for comp in pressured["competition_name"].unique():
    train, test = pressured[pressured["competition_name"] != comp], pressured[pressured["competition_name"] == comp]
    pipe = Pipeline([
        ("prep", ColumnTransformer([("num", StandardScaler(), NUMERIC),
                                    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30), CATEGORICAL)])),
        ("clf", xgb)])
    pipe.fit(train[NUMERIC + CATEGORICAL], train["retained"].astype(int))
    auc = roc_auc_score(test["retained"].astype(int), pipe.predict_proba(test[NUMERIC + CATEGORICAL])[:, 1])
    print(f"{comp:25s} AUC = {auc:.3f}")